# Comprehensive Dataset Analysis

**Student ID:** 25509225

This notebook provides comprehensive analysis of both Image Classification and Object Detection datasets, including:
- Data distribution and class balance analysis
- Sample visualization from different splits
- Image dimension analysis
- Object detection annotation analysis
- Comparative statistics report

## Part 1: Import Required Libraries

In [ ]:
import os
import sys
import json
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

# Configuration
STUDENT_ID = "25509225"
BASE_DATA_PATH = project_root / "data" / STUDENT_ID
CLASS_DATASET_PATH = BASE_DATA_PATH / "Image_Classification" / "split_dataset"
DETECTION_DATASET_PATH = BASE_DATA_PATH / "Object_Detection"

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ All dependencies imported successfully")
print(f"✓ Project root: {project_root}")
print(f"✓ Classification dataset path: {CLASS_DATASET_PATH}")
print(f"✓ Detection dataset path: {DETECTION_DATASET_PATH}")

## Helper Functions

In [ ]:
def get_image_files(path):
    """Get all image files from a directory."""
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif'}
    if not Path(path).exists():
        return []
    return [f.name for f in Path(path).iterdir() 
            if f.is_file() and f.suffix.lower() in image_extensions]


def analyze_split_dataset(split_path):
    """Analyze Image Classification split dataset."""
    stats = {
        'train': {'classes': {}, 'total': 0},
        'valid': {'classes': {}, 'total': 0},
        'test': {'classes': {}, 'total': 0}
    }
    
    for split in ['train', 'valid', 'test']:
        split_dir = split_path / split
        if not split_dir.exists():
            continue
            
        for class_dir in sorted(split_dir.iterdir()):
            if class_dir.is_dir():
                images = get_image_files(class_dir)
                class_name = class_dir.name
                stats[split]['classes'][class_name] = len(images)
                stats[split]['total'] += len(images)
    
    return stats


def get_image_dimensions(image_path):
    """Get image width, height, and aspect ratio."""
    try:
        with Image.open(image_path) as img:
            w, h = img.size
            aspect_ratio = w / h if h > 0 else 0
            return {'width': w, 'height': h, 'aspect_ratio': aspect_ratio}
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None


def collect_image_dimensions(split_path, max_samples=None):
    """Collect image dimensions from dataset."""
    dimensions = []
    count = 0
    
    for split in ['train', 'valid', 'test']:
        split_dir = split_path / split
        if not split_dir.exists():
            continue
            
        for class_dir in sorted(split_dir.iterdir()):
            if class_dir.is_dir():
                for img_file in get_image_files(class_dir):
                    if max_samples and count >= max_samples:
                        break
                    
                    img_path = class_dir / img_file
                    dims = get_image_dimensions(img_path)
                    if dims:
                        dims['split'] = split
                        dims['class'] = class_dir.name
                        dimensions.append(dims)
                        count += 1
    
    return pd.DataFrame(dimensions)


print("✓ Helper functions defined")

---

# Part 2: Image Classification Dataset Analysis

## Section 1: Load and Explore Split Dataset

In [ ]:
print("="*80)
print("IMAGE CLASSIFICATION DATASET ANALYSIS")
print("="*80)

# Verify path exists
if not CLASS_DATASET_PATH.exists():
    print(f"❌ Dataset path not found: {CLASS_DATASET_PATH}")
else:
    print(f"✓ Dataset path verified: {CLASS_DATASET_PATH}\n")
    
    # Analyze split dataset
    class_stats = analyze_split_dataset(CLASS_DATASET_PATH)
    
    print("Data Distribution:")
    print(f"{'-'*80}")
    print(f"{'Split':<15} {'Total Images':<20} {'Classes':<20}")
    print(f"{'-'*80}")
    
    total_all = 0
    for split in ['train', 'valid', 'test']:
        num_classes = len(class_stats[split]['classes'])
        num_images = class_stats[split]['total']
        total_all += num_images
        print(f"{split.capitalize():<15} {num_images:<20} {num_classes:<20}")
    
    print(f"{'-'*80}")
    print(f"{'TOTAL':<15} {total_all:<20}")
    print()
    
    # Create DataFrame for easier analysis
    class_dist_data = []
    for split in ['train', 'valid', 'test']:
        for cls, count in class_stats[split]['classes'].items():
            class_dist_data.append({'Split': split.capitalize(), 'Class': cls, 'Count': count})
    
    class_df = pd.DataFrame(class_dist_data)
    print("\nPer-Class Distribution:")
    print(class_df.to_string(index=False))

## Section 2: Visualize Class Distribution

In [ ]:
# Visualization 1: Overall Split Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
split_totals = [class_stats[split]['total'] for split in ['train', 'valid', 'test']]
colors = ['#2ecc71', '#3498db', '#e74c3c']
axes[0].bar(['Train', 'Validation', 'Test'], split_totals, color=colors)
axes[0].set_title('Image Distribution Across Splits', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Images')
for i, v in enumerate(split_totals):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(split_totals, labels=['Train', 'Validation', 'Test'], autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[1].set_title('Split Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Visualization 2: Per-Class Distribution across Splits
fig, ax = plt.subplots(figsize=(14, 8))

classes = sorted(class_stats['train']['classes'].keys())
x = np.arange(len(classes))
width = 0.25

train_counts = [class_stats['train']['classes'].get(cls, 0) for cls in classes]
valid_counts = [class_stats['valid']['classes'].get(cls, 0) for cls in classes]
test_counts = [class_stats['test']['classes'].get(cls, 0) for cls in classes]

ax.bar(x - width, train_counts, width, label='Train', color='#2ecc71')
ax.bar(x, valid_counts, width, label='Validation', color='#3498db')
ax.bar(x + width, test_counts, width, label='Test', color='#e74c3c')

ax.set_xlabel('Class', fontweight='bold')
ax.set_ylabel('Number of Images', fontweight='bold')
ax.set_title('Per-Class Distribution Across Splits', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Class distribution visualizations completed")

## Section 3: Display Sample Images

In [ ]:
# Display sample images from each class (only train split, max 10 classes)
classes_to_show = sorted(class_stats['train']['classes'].keys())[:10]  # Limit to first 10 classes

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

fig.suptitle('Sample Images from Train Split (First 10 Classes)', fontsize=16, fontweight='bold', y=0.995)

for idx, class_name in enumerate(classes_to_show):
    row = idx // 5
    col = idx % 5
    
    train_dir = CLASS_DATASET_PATH / 'train' / class_name
    
    if train_dir.exists():
        images = get_image_files(train_dir)
        if images:
            img_path = train_dir / images[0]
            try:
                img = Image.open(img_path)
                axes[row, col].imshow(img)
                axes[row, col].set_title(f'{class_name}({len(images)} images)', fontsize=10)
                axes[row, col].axis('off')
            except Exception as e:
                axes[row, col].text(0.5, 0.5, f'Error loading{class_name}',ha='center', va='center', transform=axes[row, col].transAxes)
                axes[row, col].axis('off')

# Hide any unused subplots
for idx in range(len(classes_to_show), 10):
    row = idx // 5
    col = idx % 5
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("✓ Sample images displayed")

## Section 4: Analyze Image Dimensions

Analyzing image dimensions and aspect ratios across the dataset...

In [ ]:
print("Collecting image dimensions (this may take a moment)...")

# Collect image dimensions (limit to 500 samples for speed)
dims_df = collect_image_dimensions(CLASS_DATASET_PATH, max_samples=500)

if len(dims_df) > 0:
    print(f"\n✓ Analyzed {len(dims_df)} images")
    print(f"\nImage Dimension Statistics:")
    print(f"{'-'*80}")
    print(dims_df[['width', 'height', 'aspect_ratio']].describe().to_string())
    
    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Width distribution
    axes[0, 0].hist(dims_df['width'], bins=30, color='#3498db', edgecolor='black')
    axes[0, 0].set_xlabel('Width (pixels)', fontweight='bold')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Image Width Distribution', fontweight='bold')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # Height distribution
    axes[0, 1].hist(dims_df['height'], bins=30, color='#2ecc71', edgecolor='black')
    axes[0, 1].set_xlabel('Height (pixels)', fontweight='bold')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Image Height Distribution', fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # Aspect ratio distribution
    axes[1, 0].hist(dims_df['aspect_ratio'], bins=30, color='#e74c3c', edgecolor='black')
    axes[1, 0].set_xlabel('Aspect Ratio (W/H)', fontweight='bold')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Aspect Ratio Distribution', fontweight='bold')
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # Scatter: Width vs Height
    for split in dims_df['split'].unique():
        split_data = dims_df[dims_df['split'] == split]
        axes[1, 1].scatter(split_data['width'], split_data['height'], 
                          label=split.capitalize(), alpha=0.6, s=30)
    axes[1, 1].set_xlabel('Width (pixels)', fontweight='bold')
    axes[1, 1].set_ylabel('Height (pixels)', fontweight='bold')
    axes[1, 1].set_title('Width vs Height by Split', fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Image dimension analysis completed")
else:
    print("❌ No images found to analyze")

---

# Part 3: Object Detection Dataset Analysis

## Section 5: Load and Explore Detection Dataset

In [ ]:
print("="*80)
print("OBJECT DETECTION DATASET ANALYSIS")
print("="*80)

# Check which detection format is available
detection_formats = {
    'YOLO': DETECTION_DATASET_PATH / 'yolo',
    'COCO': DETECTION_DATASET_PATH / 'coco',
}

available_formats = [fmt for fmt, path in detection_formats.items() if path.exists()]
print(f"\nAvailable formats: {', '.join(available_formats)}\n")

# Helper functions for parsing annotations
def parse_yolo_annotation(yolo_txt_path, image_width, image_height):
    """Parse YOLO format annotation."""
    objects = []
    if not yolo_txt_path.exists():
        return objects
    
    try:
        with open(yolo_txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_center = float(parts[1]) * image_width
                    y_center = float(parts[2]) * image_height
                    width = float(parts[3]) * image_width
                    height = float(parts[4]) * image_height
                    
                    objects.append({
                        'class_id': class_id,
                        'x_center': x_center,
                        'y_center': y_center,
                        'width': width,
                        'height': height,
                        'bbox_area': width * height
                    })
    except Exception as e:
        pass
    
    return objects


def parse_coco_json(json_path):
    """Parse COCO format annotations."""
    if not json_path.exists():
        return [], {}
    
    try:
        with open(json_path, 'r') as f:
            coco_data = json.load(f)
        
        annotations = []
        for ann in coco_data.get('annotations', []):
            x, y, w, h = ann['bbox']
            annotations.append({
                'class_id': ann['category_id'],
                'x': x,
                'y': y,
                'width': w,
                'height': h,
                'bbox_area': w * h,
                'image_id': ann['image_id']
            })
        
        # Map image_id to filename
        image_map = {img['id']: img['file_name'] for img in coco_data.get('images', [])}
        class_map = {cat['id']: cat['name'] for cat in coco_data.get('categories', [])}
        
        return annotations, class_map
    except Exception as e:
        print(f"Error parsing COCO: {e}")
        return [], {}

print("✓ Detection parsing functions defined")

### YOLO Format Analysis

In [ ]:
if 'YOLO' in available_formats:
    yolo_path = DETECTION_DATASET_PATH / 'yolo'
    print("="*80)
    print("YOLO DATASET STRUCTURE ANALYSIS")
    print("="*80)
    
    # 1. Directory Structure Analysis
    print("\n1. Directory Structure:")
    print("-" * 80)
    
    yolo_stats = {'train': {'images': 0, 'labels': 0}, 
                  'valid': {'images': 0, 'labels': 0}, 
                  'test': {'images': 0, 'labels': 0}}
    
    for split in ['train', 'valid', 'test']:
        split_path = yolo_path / split
        if split_path.exists():
            print(f"\n{split.upper()}:")
            images_dir = split_path / 'images'
            labels_dir = split_path / 'labels'
            
            if images_dir.exists():
                image_files = [f for f in images_dir.iterdir() 
                              if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
                yolo_stats[split]['images'] = len(image_files)
                print(f"  Images: {len(image_files)} files")
                if image_files:
                    print(f"    Sample: {image_files[0].name}")
            else:
                print(f"  Images: Directory not found")
                
            if labels_dir.exists():
                label_files = [f for f in labels_dir.iterdir() if f.is_file() and f.suffix == '.txt']
                yolo_stats[split]['labels'] = len(label_files)
                print(f"  Labels: {len(label_files)} files")
                if label_files:
                    print(f"    Sample: {label_files[0].name}")
            else:
                print(f"  Labels: Directory not found")
        else:
            print(f"\n{split.upper()}: Directory not found")
    
    # 2. Label File Format Analysis
    print("\n\n2. Label File Format Analysis:")
    print("-" * 80)
    
    train_labels_dir = yolo_path / 'train' / 'labels'
    sample_label_file = None
    
    if train_labels_dir.exists():
        for txt_file in train_labels_dir.iterdir():
            if txt_file.is_file() and txt_file.suffix == '.txt' and txt_file.stat().st_size > 0:
                sample_label_file = txt_file
                break
        
        if sample_label_file:
            print(f"\nSample label file: {sample_label_file.name}")
            print(f"File size: {sample_label_file.stat().st_size} bytes")
            
            with open(sample_label_file, 'r') as f:
                lines = f.readlines()
            
            print(f"Number of objects: {len(lines)}")
            print(f"\nFirst 5 lines (format: class_id center_x center_y width height):")
            for i, line in enumerate(lines[:5]):
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id, cx, cy, w, h = parts
                    print(f"  Line {i+1}: class={class_id}, cx={cx}, cy={cy}, w={w}, h={h}")
        else:
            print("\nNo label files with content found")
    
    # 3. Class Distribution Analysis
    print("\n\n3. Class Distribution in Train Set:")
    print("-" * 80)
    
    from collections import Counter
    class_counter = Counter()
    total_objects = 0
    files_with_objects = 0
    
    if train_labels_dir.exists():
        for txt_file in train_labels_dir.iterdir():
            if txt_file.is_file() and txt_file.suffix == '.txt' and txt_file.stat().st_size > 0:
                files_with_objects += 1
                with open(txt_file, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 1:
                            class_id = int(parts[0])
                            class_counter[class_id] += 1
                            total_objects += 1
    
    print(f"Total label files: {yolo_stats['train']['labels']}")
    print(f"Files with objects: {files_with_objects}")
    print(f"Total objects: {total_objects}")
    print(f"\nClass distribution:")
    for class_id in sorted(class_counter.keys()):
        count = class_counter[class_id]
        percentage = (count / total_objects * 100) if total_objects > 0 else 0
        print(f"  Class {class_id}: {count:6d} objects ({percentage:5.1f}%)")
    
    # 4. Image-Label Correspondence
    print("\n\n4. Image-Label Correspondence:")
    print("-" * 80)
    
    train_images_dir = yolo_path / 'train' / 'images'
    if train_images_dir.exists() and train_labels_dir.exists():
        image_names = set([f.stem for f in train_images_dir.iterdir() 
                          if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
        label_names = set([f.stem for f in train_labels_dir.iterdir() 
                          if f.is_file() and f.suffix == '.txt'])
        
        matched = image_names & label_names
        only_images = image_names - label_names
        only_labels = label_names - image_names
        
        # Count empty labels
        empty_labels = 0
        non_empty_labels = 0
        for txt_file in train_labels_dir.iterdir():
            if txt_file.is_file() and txt_file.suffix == '.txt':
                if txt_file.stat().st_size == 0:
                    empty_labels += 1
                else:
                    non_empty_labels += 1
        
        print(f"Images: {len(image_names)}")
        print(f"Labels: {len(label_names)}")
        print(f"Matched (both image and label exist): {len(matched)}")
        print(f"  - Labels with objects: {non_empty_labels}")
        print(f"  - Empty labels (no objects): {empty_labels}")
        print(f"Only images (no label file): {len(only_images)}")
        print(f"Only labels (no image file): {len(only_labels)}")
        
        if only_images:
            print(f"\nSample images without labels: {list(only_images)[:5]}")
        if only_labels:
            print(f"\nSample labels without images: {list(only_labels)[:5]}")
    
    # 5. Visualize Class Distribution
    print("\n\n5. Class Distribution Visualization:")
    print("-" * 80)
    
    if class_counter:
        fig, ax = plt.subplots(figsize=(12, 6))
        classes = sorted(class_counter.keys())
        counts = [class_counter[c] for c in classes]
        colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
        
        bars = ax.bar([f'Class {c}' for c in classes], counts, color=colors[:len(classes)])
        ax.set_xlabel('Class ID', fontweight='bold')
        ax.set_ylabel('Number of Objects', fontweight='bold')
        ax.set_title('YOLO Train Set - Class Distribution', fontsize=14, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        
        # Add value labels on bars
        for bar, count in zip(bars, counts):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{count}\n({count/total_objects*100:.1f}%)',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
    
    print("\n✓ YOLO dataset analysis completed")
    print(f"\nAnnotation Format: class_id center_x center_y width height (normalized 0-1)")
    
    # 6. Display Sample Images with Bounding Boxes
    print("\n\n6. Sample Images with Bounding Boxes:")
    print("-" * 80)
    
    def parse_yolo_label(label_path):
        """Parse YOLO label file and return list of annotations."""
        annotations = []
        if not label_path.exists() or label_path.stat().st_size == 0:
            return annotations
        
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id = int(parts[0])
                    center_x = float(parts[1])
                    center_y = float(parts[2])
                    width = float(parts[3])
                    height = float(parts[4])
                    annotations.append({
                        'class_id': class_id,
                        'center_x': center_x,
                        'center_y': center_y,
                        'width': width,
                        'height': height
                    })
        return annotations
    
    def draw_yolo_image_with_boxes(image_path, label_path, ax):
        """Draw image with YOLO format bounding boxes."""
        from matplotlib.patches import Rectangle
        
        try:
            img = Image.open(image_path)
            img_width, img_height = img.size
            
            ax.imshow(img)
            
            # Parse labels
            annotations = parse_yolo_label(label_path)
            
            # Draw each bounding box
            colors_map = {0: 'red', 1: 'blue', 2: 'green', 3: 'orange', 4: 'purple'}
            
            for ann in annotations:
                # Convert normalized coordinates to pixel coordinates
                x_center = ann['center_x'] * img_width
                y_center = ann['center_y'] * img_height
                width = ann['width'] * img_width
                height = ann['height'] * img_height
                
                # Calculate top-left corner
                x_min = x_center - width / 2
                y_min = y_center - height / 2
                
                # Draw rectangle
                color = colors_map.get(ann['class_id'], 'yellow')
                rect = Rectangle((x_min, y_min), width, height, 
                               linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                
                # Add class label
                ax.text(x_min, y_min - 2, str(ann['class_id']), 
                       color='white', fontsize=7, fontweight='bold',
                       bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))
            
            num_objects = len(annotations)
            ax.set_title(f'{image_path.name} {num_objects} objects', 
                        fontsize=11, fontweight='bold')
            ax.axis('off')
            
        except Exception as e:
            ax.text(0.5, 0.5, f'Error: {str(e)}', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
    
    # Select one sample from each split (choose images with many objects)
    splits_to_show = ['train', 'valid', 'test']
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    fig.suptitle('YOLO Dataset - Sample Images with Many Objects', 
                fontsize=16, fontweight='bold', y=1.02)
    
    for idx, split in enumerate(splits_to_show):
        images_dir = yolo_path / split / 'images'
        labels_dir = yolo_path / split / 'labels'
        
        if images_dir.exists() and labels_dir.exists():
            # Get all image-label pairs with object counts
            image_label_pairs = []
            
            for img_file in images_dir.iterdir():
                if img_file.is_file() and img_file.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                    label_file = labels_dir / f"{img_file.stem}.txt"
                    if label_file.exists() and label_file.stat().st_size > 0:
                        # Count objects in this label file
                        with open(label_file, 'r') as f:
                            num_objects = sum(1 for line in f if line.strip())
                        image_label_pairs.append((img_file, label_file, num_objects))
            
            if image_label_pairs:
                # Sort by number of objects (descending) and select top one
                image_label_pairs.sort(key=lambda x: x[2], reverse=True)
                selected_image, selected_label, num_objs = image_label_pairs[0]
                
                draw_yolo_image_with_boxes(selected_image, selected_label, axes[idx])
                print(f"  {split.capitalize()}: Selected '{selected_image.name}' with {num_objs} objects")
            else:
                axes[idx].text(0.5, 0.5, f'No labeled images in {split}', 
                              ha='center', va='center', transform=axes[idx].transAxes, fontsize=10)
                axes[idx].axis('off')
        else:
            axes[idx].text(0.5, 0.5, f'{split} directory not found', 
                          ha='center', va='center', transform=axes[idx].transAxes, fontsize=10)
            axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    print("\n✓ Displayed sample images with bounding boxes (selected images with most objects)")
    
else:
    print("\n⚠ YOLO dataset not found")

### COCO Format Analysis

In [ ]:
if 'COCO' in available_formats:
    coco_path = DETECTION_DATASET_PATH / 'coco'
    print("\nCOCO Dataset Analysis")
    print(f"{'-'*80}")
    
    coco_stats = {'train': {}, 'valid': {}, 'test': {}}
    all_coco_annotations = []
    
    for split in ['train', 'valid', 'test']:
        json_file = coco_path / split / f'{split}_annotations.json'
        if json_file.exists():
            annotations, class_map = parse_coco_json(json_file)
            all_coco_annotations.extend(annotations)
            
            # COCO has images directly in split directory
            image_count = len([f for f in (coco_path / split).iterdir() 
                             if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
            
            coco_stats[split] = {
                'images': image_count,
                'annotations': len(annotations),
                'classes': class_map
            }
    
    # Print statistics
    print("Split Distribution:")
    for split in ['train', 'valid', 'test']:
        if split in coco_stats and coco_stats[split]:
            img_count = coco_stats[split].get('images', 0)
            ann_count = coco_stats[split].get('annotations', 0)
            if img_count > 0:
                objs_per_img = ann_count / img_count
                print(f"  {split.capitalize():<12}: {img_count:>6} images, {ann_count:>6} objects ({objs_per_img:>5.1f} obj/img)")
    
    # Show categories
    if coco_stats['train'] and coco_stats['train']['classes']:
        print(f"\nCategories ({len(coco_stats['train']['classes'])} total):")
        for cat_id, cat_name in sorted(coco_stats['train']['classes'].items()):
            print(f"  {cat_id}: {cat_name}")
    
    print("\nAnnotation Format: bbox=[x, y, width, height] (pixel coordinates)")
    print("✓ COCO dataset analyzed")
    
    # Display sample images from COCO train split
    print("\nDisplaying sample images from COCO train split...")
    coco_train_dir = coco_path / 'train'
    if coco_train_dir.exists():
        sample_images = sorted([f for f in coco_train_dir.iterdir() 
                               if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png'}])[:5]
        
        if len(sample_images) > 0:
            fig, axes = plt.subplots(1, min(5, len(sample_images)), figsize=(20, 4))
            if len(sample_images) == 1:
                axes = [axes]
            
            fig.suptitle('COCO Train Split - Sample Images', fontsize=14, fontweight='bold')
            
            for idx, img_file in enumerate(sample_images):
                try:
                    img = Image.open(img_file)
                    axes[idx].imshow(img)
                    axes[idx].set_title(f'{img_file.name}', fontsize=9)
                    axes[idx].axis('off')
                except Exception as e:
                    axes[idx].text(0.5, 0.5, 'Error', ha='center', va='center', 
                                  transform=axes[idx].transAxes)
                    axes[idx].axis('off')
            
            plt.tight_layout()
            plt.show()
            print(f"✓ Displayed {len(sample_images)} sample images")
        else:
            print("⚠ No images found in COCO train split")
    else:
        print("⚠ COCO train directory not found")
else:
    print("\n⚠ COCO dataset not found")